# **Final Solar Deliverables (2021 and 2024)**

Last Updated: 04/25/26

This notebook creates and packages the final deliverables for:

1. **PowerGenome candidate solar sites**
   - available candidate sites mapped to counties
   - county summary
   - state summary

2. **Installed solar tables**
   - EIA 2021 and 2024 by county/state
   - GEM 2021 and 2024 by county/state

3. **Comparison tables**
   - EIA vs GEM for 2021 and 2024 at the county and state level

**Note:** 2024 is used in place of 2025 because final annual EIA data are available through 2024.


In [4]:
from pathlib import Path
import shutil
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

In [5]:
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent

COUNTY_GEOJSON = REPO_ROOT / "county_layer_for_gui.geojson"

PG_DIR = REPO_ROOT / "powergenome_wecc_solar_2030"
PG_RESOURCE_DIR = PG_DIR / "resource_groups"
PG_SITE_MAP_DIR = PG_DIR / "site_maps"
PG_OUTPUT_DIR = PG_DIR / "outputs"

INSTALLED_DIR = REPO_ROOT / "solar_installed_outputs"

FINAL_DIR = NOTEBOOK_DIR / "final_deliverables"
FINAL_CANDIDATE_DIR = FINAL_DIR / "candidate_sites"
FINAL_INSTALLED_DIR = FINAL_DIR / "installed_solar"
FINAL_COMPARE_DIR = FINAL_DIR / "comparisons"

for d in [FINAL_DIR, FINAL_CANDIDATE_DIR, FINAL_INSTALLED_DIR, FINAL_COMPARE_DIR, PG_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

paths_to_check = {
    "notebook_dir": NOTEBOOK_DIR,
    "repo_root": REPO_ROOT,
    "county_geojson": COUNTY_GEOJSON,
    "pg_solar_parquet": PG_RESOURCE_DIR / "solar_lcoe_wecc_solar_2030.parquet",
    "pg_helper_csv": PG_SITE_MAP_DIR / "solar_lcoe_wecc_14_zone.csv",
    "installed_outputs_dir": INSTALLED_DIR,
    "pg_output_dir": PG_OUTPUT_DIR,
    "final_dir": FINAL_DIR,
}

for name, p in paths_to_check.items():
    print(f"{name}: {p.exists()} -> {p}")

notebook_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/notebooks
repo_root: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores
county_geojson: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/county_layer_for_gui.geojson
pg_solar_parquet: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/powergenome_wecc_solar_2030/resource_groups/solar_lcoe_wecc_solar_2030.parquet
pg_helper_csv: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/powergenome_wecc_solar_2030/site_maps/solar_lcoe_wecc_14_zone.csv
installed_outputs_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/solar_installed_outputs
pg_output_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/powergenome_wecc_solar_2030/outputs
final_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/notebooks/final_deliverables


## Part 1 — Candidate solar sites on the county layer

This section:
- loads the PowerGenome candidate solar file
- merges in longitude/latitude from the WECC 14-zone helper file
- filters to **available** sites using `anyQual == 1`
- spatially joins candidate sites to the county layer
- writes final site-, county-, and state-level candidate solar deliverables


In [6]:
solar_df = pd.read_parquet(PG_RESOURCE_DIR / "solar_lcoe_wecc_solar_2030.parquet")
helper_df = pd.read_csv(PG_SITE_MAP_DIR / "solar_lcoe_wecc_14_zone.csv", low_memory=False)
helper_df = helper_df.rename(columns={"CPA_ID": "cpa_id"})

solar_ids = set(solar_df["cpa_id"].dropna().unique())
helper_ids = set(helper_df["cpa_id"].dropna().unique())
overlap = solar_ids & helper_ids

print("solar_df shape:", solar_df.shape)
print("helper_df shape:", helper_df.shape)
print("solar unique ids:", len(solar_ids))
print("helper unique ids:", len(helper_ids))
print("overlap ids:", len(overlap))
print("solar match rate:", len(overlap) / len(solar_ids) if solar_ids else None)

coord_cols = [c for c in ["cpa_id", "longitude", "latitude", "metro_name", "ipm_region"] if c in helper_df.columns]
display(helper_df[coord_cols].head())

solar_df shape: (159408, 16)
helper_df shape: (402688, 36)
solar unique ids: 159408
helper unique ids: 402688
overlap ids: 157172
solar match rate: 0.9859731004717455


,cpa_id,longitude,latitude,metro_name,ipm_region
0,1,-122.304630,48.970628,"Bellingham, WA",WECC_PNW
1,2,-122.366856,48.938453,"Bellingham, WA",WECC_PNW
2,3,-121.965700,48.989229,"Seattle-Tacoma-Bellevue, WA",WECC_PNW
3,4,-122.131989,48.911972,"Seattle-Tacoma-Bellevue, WA",WECC_PNW
4,6,-122.137923,48.875907,"Seattle-Tacoma-Bellevue, WA",WECC_PNW


In [7]:
merged = solar_df.merge(
    helper_df[[c for c in ["cpa_id", "longitude", "latitude"] if c in helper_df.columns]],
    on="cpa_id",
    how="left"
)

print("missing longitude share:", merged["longitude"].isna().mean())
print("missing latitude share:", merged["latitude"].isna().mean())

available_sites = merged[merged["anyQual"] == 1].copy()
print("available candidate site count:", len(available_sites))

display(available_sites.head())

missing longitude share: 0.014026899528254541
missing latitude share: 0.014026899528254541
available candidate site count: 70264


,cpa_id,msa_id,cpa_mw,cf,path,path_len,lcoe,interconnect_capex_mw,total_interconnect_km,region,msa_name,anyQual,m_popden,exFacil,plFacil,capacity_mw,longitude,latitude
0,219261,19700,34.747166,0.319071,"[219261,300657]",2.0,22.620781,17228.830078,1.103553,NM1,"Deming, NM",1,5.757576,0,0,34.747166,-107.791199,32.304256
1,212932,37220,89.863327,0.330012,"[212932,310177]",2.0,22.628979,63279.906250,15.071069,NV1,"Pahrump, NV",1,0.000000,0,0,89.863327,-115.831334,36.174793
2,107220,29420,111.978813,0.318505,"[107220,303516]",2.0,22.663147,17356.224609,0.957107,AZ1,"Lake Havasu City-Kingman, AZ",1,0.276596,0,0,111.978813,-114.141078,35.060884
3,219262,19700,37.833187,0.319071,"[219262,300657]",2.0,22.682138,20832.226562,2.707107,NM1,"Deming, NM",1,9.210526,0,0,37.833187,-107.753521,32.310589
4,212941,37220,70.248444,0.330012,"[212941,310177]",2.0,22.682661,66540.640625,16.399496,NV1,"Pahrump, NV",1,0.000000,0,0,70.248444,-115.846086,36.245512


In [8]:
counties = gpd.read_file(COUNTY_GEOJSON).to_crs("EPSG:4326").copy()
counties = counties.rename(columns={
    "NAME": "county_name_geo",
    "STATE_NAME": "state_name_geo",
    "STUSPS": "state_abbrev_geo",
    "GEOID": "county_fips_geo",
})

sites_gdf = gpd.GeoDataFrame(
    available_sites.copy(),
    geometry=gpd.points_from_xy(available_sites["longitude"], available_sites["latitude"]),
    crs="EPSG:4326"
)

sites_with_counties = gpd.sjoin(
    sites_gdf,
    counties[["county_name_geo", "state_name_geo", "state_abbrev_geo", "county_fips_geo", "geometry"]],
    how="left",
    predicate="within"
)

print("missing county_fips share:", sites_with_counties["county_fips_geo"].isna().mean())
display(sites_with_counties.head())

missing county_fips share: 0.0002988728224980075


,cpa_id,msa_id,cpa_mw,cf,path,path_len,lcoe,interconnect_capex_mw,total_interconnect_km,region,msa_name,anyQual,m_popden,exFacil,plFacil,capacity_mw,longitude,latitude,geometry,index_right,county_name_geo,state_name_geo,state_abbrev_geo,county_fips_geo
0,219261,19700,34.747166,0.319071,"[219261,300657]",2.0,22.620781,17228.830078,1.103553,NM1,"Deming, NM",1,5.757576,0,0,34.747166,-107.791199,32.304256,POINT (-107.7912 32.30426),431.0,Luna,New Mexico,NM,35029
1,212932,37220,89.863327,0.330012,"[212932,310177]",2.0,22.628979,63279.906250,15.071069,NV1,"Pahrump, NV",1,0.000000,0,0,89.863327,-115.831334,36.174793,POINT (-115.83133 36.17479),799.0,Clark,Nevada,NV,32003
2,107220,29420,111.978813,0.318505,"[107220,303516]",2.0,22.663147,17356.224609,0.957107,AZ1,"Lake Havasu City-Kingman, AZ",1,0.276596,0,0,111.978813,-114.141078,35.060884,POINT (-114.14108 35.06088),6.0,Mohave,Arizona,AZ,04015
3,219262,19700,37.833187,0.319071,"[219262,300657]",2.0,22.682138,20832.226562,2.707107,NM1,"Deming, NM",1,9.210526,0,0,37.833187,-107.753521,32.310589,POINT (-107.75352 32.31059),431.0,Luna,New Mexico,NM,35029
4,212941,37220,70.248444,0.330012,"[212941,310177]",2.0,22.682661,66540.640625,16.399496,NV1,"Pahrump, NV",1,0.000000,0,0,70.248444,-115.846086,36.245512,POINT (-115.84609 36.24551),799.0,Clark,Nevada,NV,32003


In [9]:
site_cols = [
    "cpa_id", "region", "msa_id", "msa_name",
    "capacity_mw", "lcoe", "interconnect_capex_mw",
    "longitude", "latitude",
    "county_name_geo", "state_name_geo", "state_abbrev_geo", "county_fips_geo"
]
site_cols = [c for c in site_cols if c in sites_with_counties.columns]

candidate_sites_final = sites_with_counties[site_cols].rename(columns={
    "county_name_geo": "county_name",
    "state_name_geo": "state_name",
    "state_abbrev_geo": "state_abbrev",
    "county_fips_geo": "county_fips",
}).copy()

candidate_sites_final.to_csv(
    FINAL_CANDIDATE_DIR / "powergenome_candidate_solar_sites_with_counties.csv",
    index=False
)

candidate_sites_final.to_csv(
    PG_OUTPUT_DIR / "powergenome_candidate_solar_sites_with_counties.csv",
    index=False
)

display(candidate_sites_final.head())

,cpa_id,region,msa_id,msa_name,capacity_mw,lcoe,interconnect_capex_mw,longitude,latitude,county_name,state_name,state_abbrev,county_fips
0,219261,NM1,19700,"Deming, NM",34.747166,22.620781,17228.830078,-107.791199,32.304256,Luna,New Mexico,NM,35029
1,212932,NV1,37220,"Pahrump, NV",89.863327,22.628979,63279.906250,-115.831334,36.174793,Clark,Nevada,NV,32003
2,107220,AZ1,29420,"Lake Havasu City-Kingman, AZ",111.978813,22.663147,17356.224609,-114.141078,35.060884,Mohave,Arizona,AZ,04015
3,219262,NM1,19700,"Deming, NM",37.833187,22.682138,20832.226562,-107.753521,32.310589,Luna,New Mexico,NM,35029
4,212941,NV1,37220,"Pahrump, NV",70.248444,22.682661,66540.640625,-115.846086,36.245512,Clark,Nevada,NV,32003


In [10]:
county_candidate_summary = (
    sites_with_counties.groupby(
        ["state_abbrev_geo", "state_name_geo", "county_name_geo", "county_fips_geo"],
        dropna=False
    )
    .agg(
        candidate_site_count=("cpa_id", "count"),
        total_candidate_mw=("capacity_mw", "sum"),
        avg_lcoe=("lcoe", "mean"),
        min_lcoe=("lcoe", "min"),
    )
    .reset_index()
    .rename(columns={
        "state_abbrev_geo": "state_abbrev",
        "state_name_geo": "state_name",
        "county_name_geo": "county_name",
        "county_fips_geo": "county_fips",
    })
    .sort_values(["avg_lcoe", "total_candidate_mw"], ascending=[True, False])
)

county_candidate_summary.to_csv(
    FINAL_CANDIDATE_DIR / "powergenome_candidate_solar_by_county.csv",
    index=False
)

county_candidate_summary.to_csv(
    PG_OUTPUT_DIR / "powergenome_candidate_solar_by_county.csv",
    index=False
)

display(county_candidate_summary.head(20))

,state_abbrev,state_name,county_name,county_fips,candidate_site_count,total_candidate_mw,avg_lcoe,min_lcoe
148,NM,New Mexico,Doña Ana,35013,1,7.371854,23.743225,23.743225
166,NM,New Mexico,Sandoval,35043,2,235.454361,23.753458,23.469570
241,TX,Texas,El Paso,48141,96,9199.701172,24.140999,22.936644
186,NV,Nevada,Storey,32029,1,66.805099,24.277304,24.277304
7,AZ,Arizona,Maricopa,04013,8,615.391663,24.533176,23.956079
167,NM,New Mexico,Santa Fe,35049,2,169.783264,25.014753,24.439564
13,AZ,Arizona,Yavapai,04025,1,6.862651,25.205509,25.205509
11,AZ,Arizona,Pinal,04021,9,859.821106,25.479954,24.351622
156,NM,New Mexico,Los Alamos,35028,9,283.266998,25.866213,25.514870
23,CA,California,Napa,06055,1,8.437500,25.986464,25.986464


In [11]:
state_candidate_summary = (
    sites_with_counties.groupby(
        ["state_abbrev_geo", "state_name_geo"],
        dropna=False
    )
    .agg(
        candidate_site_count=("cpa_id", "count"),
        total_candidate_mw=("capacity_mw", "sum"),
        avg_lcoe=("lcoe", "mean"),
        min_lcoe=("lcoe", "min"),
    )
    .reset_index()
    .rename(columns={
        "state_abbrev_geo": "state_abbrev",
        "state_name_geo": "state_name",
    })
    .sort_values(["avg_lcoe", "total_candidate_mw"], ascending=[True, False])
)

state_candidate_summary.to_csv(
    FINAL_CANDIDATE_DIR / "powergenome_candidate_solar_by_state.csv",
    index=False
)

state_candidate_summary.to_csv(
    PG_OUTPUT_DIR / "powergenome_candidate_solar_by_state.csv",
    index=False
)

display(state_candidate_summary.head(20))

,state_abbrev,state_name,candidate_site_count,total_candidate_mw,avg_lcoe,min_lcoe
7,NE,Nebraska,3,3.516348e+02,28.522684,26.973570
11,SD,South Dakota,799,9.548784e+04,28.687639,25.614031
8,NM,New Mexico,13899,1.527572e+06,30.999367,22.620781
5,MT,Montana,5030,6.377318e+05,31.147667,25.974783
0,AZ,Arizona,6935,7.387656e+05,31.781080,22.663147
12,TX,Texas,7540,9.493598e+05,32.857624,22.936644
1,CA,California,1683,1.160608e+05,33.421646,24.778572
15,WY,Wyoming,11960,1.409845e+06,33.656410,25.057421
3,ID,Idaho,32,2.658237e+03,35.629822,26.578100
4,KS,Kansas,1,7.087500e+01,35.834446,35.834446


## Part 2 — Installed solar deliverables

This section copies the final installed-solar outputs into a single `final_deliverables/` folder.

Expected source files:
- EIA 2021 and 2024 by county/state
- GEM 2021 and 2024 by county/state
- EIA vs GEM comparison tables for 2021 and 2024


In [12]:
installed_files = [
    "eia_2021_solar_by_county.csv",
    "eia_2024_solar_by_county.csv",
    "eia_2021_solar_by_state.csv",
    "eia_2024_solar_by_state.csv",
    "gem_2021_solar_by_county.csv",
    "gem_2024_solar_by_county.csv",
    "gem_2021_solar_by_state.csv",
    "gem_2024_solar_by_state.csv",
]

comparison_files = [
    "compare_eia_vs_gem_state_2021.csv",
    "compare_eia_vs_gem_state_2024.csv",
    "compare_eia_vs_gem_county_2021.csv",
    "compare_eia_vs_gem_county_2024.csv",
]

missing_files = []

for fname in installed_files:
    src = INSTALLED_DIR / fname
    dst = FINAL_INSTALLED_DIR / fname
    if src.exists():
        shutil.copy2(src, dst)
    else:
        missing_files.append(str(src))

for fname in comparison_files:
    src = INSTALLED_DIR / fname
    dst = FINAL_COMPARE_DIR / fname
    if src.exists():
        shutil.copy2(src, dst)
    else:
        missing_files.append(str(src))

print("Missing files:")
for f in missing_files:
    print(" -", f)

print("\nInstalled files copied:")
for p in sorted(FINAL_INSTALLED_DIR.glob("*.csv")):
    print(" -", p.name)

print("\nComparison files copied:")
for p in sorted(FINAL_COMPARE_DIR.glob("*.csv")):
    print(" -", p.name)

Missing files:

Installed files copied:
 - eia_2021_solar_by_county.csv
 - eia_2021_solar_by_state.csv
 - eia_2024_solar_by_county.csv
 - eia_2024_solar_by_state.csv
 - gem_2021_solar_by_county.csv
 - gem_2021_solar_by_state.csv
 - gem_2024_solar_by_county.csv
 - gem_2024_solar_by_state.csv

Comparison files copied:
 - compare_eia_vs_gem_county_2021.csv
 - compare_eia_vs_gem_county_2024.csv
 - compare_eia_vs_gem_state_2021.csv
 - compare_eia_vs_gem_state_2024.csv


## Part 3 — Quick previews


In [13]:
print("Top candidate solar counties:")
display(county_candidate_summary.head(15))

print("Top candidate solar states:")
display(state_candidate_summary.head(15))

eia_2024_state_path = FINAL_INSTALLED_DIR / "eia_2024_solar_by_state.csv"
gem_2024_state_path = FINAL_INSTALLED_DIR / "gem_2024_solar_by_state.csv"

if eia_2024_state_path.exists():
    print("Top EIA 2024 states:")
    display(pd.read_csv(eia_2024_state_path).head(15))

if gem_2024_state_path.exists():
    print("Top GEM 2024 states:")
    display(pd.read_csv(gem_2024_state_path).head(15))

Top candidate solar counties:


,state_abbrev,state_name,county_name,county_fips,candidate_site_count,total_candidate_mw,avg_lcoe,min_lcoe
148,NM,New Mexico,Doña Ana,35013,1,7.371854,23.743225,23.743225
166,NM,New Mexico,Sandoval,35043,2,235.454361,23.753458,23.469570
241,TX,Texas,El Paso,48141,96,9199.701172,24.140999,22.936644
186,NV,Nevada,Storey,32029,1,66.805099,24.277304,24.277304
7,AZ,Arizona,Maricopa,04013,8,615.391663,24.533176,23.956079
167,NM,New Mexico,Santa Fe,35049,2,169.783264,25.014753,24.439564
13,AZ,Arizona,Yavapai,04025,1,6.862651,25.205509,25.205509
11,AZ,Arizona,Pinal,04021,9,859.821106,25.479954,24.351622
156,NM,New Mexico,Los Alamos,35028,9,283.266998,25.866213,25.514870
23,CA,California,Napa,06055,1,8.437500,25.986464,25.986464


Top candidate solar states:


,state_abbrev,state_name,candidate_site_count,total_candidate_mw,avg_lcoe,min_lcoe
7,NE,Nebraska,3,3.516348e+02,28.522684,26.973570
11,SD,South Dakota,799,9.548784e+04,28.687639,25.614031
8,NM,New Mexico,13899,1.527572e+06,30.999367,22.620781
5,MT,Montana,5030,6.377318e+05,31.147667,25.974783
0,AZ,Arizona,6935,7.387656e+05,31.781080,22.663147
12,TX,Texas,7540,9.493598e+05,32.857624,22.936644
1,CA,California,1683,1.160608e+05,33.421646,24.778572
15,WY,Wyoming,11960,1.409845e+06,33.656410,25.057421
3,ID,Idaho,32,2.658237e+03,35.629822,26.578100
4,KS,Kansas,1,7.087500e+01,35.834446,35.834446


Top EIA 2024 states:


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
0,CA,California,1044,22558.3,21.607567
1,TX,Texas,186,22465.6,120.782796
2,FL,Florida,193,10951.9,56.745596
3,NC,North Carolina,793,6783.1,8.553720
4,AZ,Arizona,147,5336.2,36.300680
5,NV,Nevada,102,5285.1,51.814706
6,GA,Georgia,153,5013.4,32.767320
7,VA,Virginia,130,4822.7,37.097692
8,OH,Ohio,63,3238.0,51.396825
9,IL,Illinois,218,2953.6,13.548624


Top GEM 2024 states:


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
0,TX,Texas,195,25073.3,128.581026
1,CA,California,983,22588.8,22.979451
2,FL,Florida,195,11027.1,56.549231
3,NC,North Carolina,760,6767.8,8.905000
4,AZ,Arizona,126,5302.6,42.084127
5,NV,Nevada,100,5297.6,52.976000
6,GA,Georgia,149,5011.2,33.632215
7,VA,Virginia,128,4856.2,37.939062
8,OH,Ohio,63,3086.6,48.993651
9,IL,Illinois,215,2956.5,13.751163


In [14]:
readme_text = '''# Final Deliverables

## Candidate Solar Sites
- `candidate_sites/powergenome_candidate_solar_sites_with_counties.csv`
- `candidate_sites/powergenome_candidate_solar_by_county.csv`
- `candidate_sites/powergenome_candidate_solar_by_state.csv`

These files contain PowerGenome candidate solar sites from the WECC workflow, mapped to counties using the county layer and the `solar_lcoe_wecc_14_zone.csv` helper file.

## Installed Solar Tables
- EIA 2021 and 2024 by county/state
- GEM 2021 and 2024 by county/state

## Comparison Tables
- EIA vs GEM comparisons for 2021 and 2024 at the county and state level

## Notes
- 2024 is used in place of 2025 because final annual EIA data are available through 2024.
- County-level candidate-site mapping used the WECC 14-zone solar coordinate helper file.
'''

(FINAL_DIR / "README_final_deliverables.md").write_text(readme_text)
print((FINAL_DIR / "README_final_deliverables.md").read_text())

# Final Deliverables

## Candidate Solar Sites
- `candidate_sites/powergenome_candidate_solar_sites_with_counties.csv`
- `candidate_sites/powergenome_candidate_solar_by_county.csv`
- `candidate_sites/powergenome_candidate_solar_by_state.csv`

These files contain PowerGenome candidate solar sites from the WECC workflow, mapped to counties using the county layer and the `solar_lcoe_wecc_14_zone.csv` helper file.

## Installed Solar Tables
- EIA 2021 and 2024 by county/state
- GEM 2021 and 2024 by county/state

## Comparison Tables
- EIA vs GEM comparisons for 2021 and 2024 at the county and state level

## Notes
- 2024 is used in place of 2025 because final annual EIA data are available through 2024.
- County-level candidate-site mapping used the WECC 14-zone solar coordinate helper file.



In [15]:
print("Final deliverables structure:")

for folder in [FINAL_CANDIDATE_DIR, FINAL_INSTALLED_DIR, FINAL_COMPARE_DIR]:
    print(f"\n{folder.relative_to(ROOT)}/")
    for p in sorted(folder.glob("*")):
        print(" -", p.name)

Final deliverables structure:

final_deliverables/candidate_sites/
 - powergenome_candidate_solar_by_county.csv
 - powergenome_candidate_solar_by_state.csv
 - powergenome_candidate_solar_sites_with_counties.csv

final_deliverables/installed_solar/
 - eia_2021_solar_by_county.csv
 - eia_2021_solar_by_state.csv
 - eia_2024_solar_by_county.csv
 - eia_2024_solar_by_state.csv
 - gem_2021_solar_by_county.csv
 - gem_2021_solar_by_state.csv
 - gem_2024_solar_by_county.csv
 - gem_2024_solar_by_state.csv

final_deliverables/comparisons/
 - compare_eia_vs_gem_county_2021.csv
 - compare_eia_vs_gem_county_2024.csv
 - compare_eia_vs_gem_state_2021.csv
 - compare_eia_vs_gem_state_2024.csv
